# Analisi Esplorativa dei Dati: Approccio Giornaliero 🇮🇹 [🇬🇧](01_eda_day_approach.ipynb)

## Previsione del Rischio di Infortunio nei Runner — Replica di Lovdal et al. (2021)

Questo notebook esplora il **dataset di carico di allenamento a livello giornaliero** utilizzato per prevedere il rischio di infortunio nei runner agonisti. Il dataset contiene snapshot giornalieri di allenamento di **74 runner olandesi d'élite** dal 2012 al 2019.

Ogni osservazione rappresenta una **finestra mobile di 7 giorni** con 10 metriche di allenamento, creando una rappresentazione a 70 feature della storia recente di allenamento dell'atleta. La variabile target è binaria: **infortunio (1)** o **nessun infortunio (0)**.

### Domande chiave di questa analisi

1. **Quanto è grave lo sbilanciamento delle classi**, e varia tra gli atleti?
2. **Cosa rivelano le distribuzioni del carico di allenamento** sulla popolazione?
3. **Come codifica il valore sentinel (−0.01)** i giorni di riposo, e quanto è prevalente?
4. **Quali feature mostrano il segnale più forte** per il rischio di infortunio?
5. **Quali pattern emergono** nella finestra di lookback di 7 giorni?

Ogni risultato in questo notebook informa direttamente una decisione di modellazione a valle — dalla scelta delle metriche alla strategia di class weighting.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# Ensure src/ is importable regardless of the notebook working directory
def _find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    return start

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    ATHLETE_ID_COL,
    DATE_COL,
    DAY_FEATURES,
    INJURY_COL,
    N_DAY_BLOCKS,
    RANDOM_SEED,
    SENTINEL_VALUE,
)
from src.data_loading import get_feature_columns, load_day_data
from src.preprocessing.day_preprocessor import handle_sentinel_values
from src.utils.logging_config import setup_logging
from src.utils.plotting import INJURY_PALETTE, PALETTE, save_figure, set_style
from src.utils.reproducibility import set_global_seed

# --- Reproducibility & style ---
set_global_seed(RANDOM_SEED)
setup_logging()
set_style()

## 1. Caricamento Dati e Ispezione delle Dimensioni

Carichiamo il CSV raw dell'approccio giornaliero usando la nostra funzione validata `load_day_data()`, che rinomina le intestazioni originali delle colonne (es. `"nr. sessions.1"`) in una convenzione pulita `day_{block}_{feature}` e valida lo schema.

In [ ]:
df = load_day_data()
feature_cols = get_feature_columns(df)

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Feature columns: {len(feature_cols)} (7 blocks × 10 features)")
print(f"Metadata columns: {ATHLETE_ID_COL}, {INJURY_COL}, {DATE_COL}")
print(f"\nData types:\n{df.dtypes.value_counts().to_string()}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
df.head()

In [ ]:
df.describe()

### Interpretazione della struttura dati

Il dataset utilizza un design a **finestra mobile** (rolling window):
- **day_0** = il giorno corrente (più vicino al potenziale evento infortunio)
- **day_6** = sei giorni fa (più distante nella finestra di lookback)

Ogni blocco di feature cattura il carico di allenamento a distanza temporale crescente. Questo è analogo a ciò che un allenatore esaminerebbe — la settimana di allenamento precedente — per valutare carico cumulativo, fatica e pattern di recupero.

Le 10 feature per giorno coprono:
- **Volume**: `nr_sessions`, `total_km`
- **Zone di intensità**: `km_z3_4` (soglia), `km_z5_t1_t2` (VO2max/anaerobica), `km_sprinting`
- **Cross-training**: `strength_training`, `hours_alternative`
- **Misure soggettive**: `perceived_exertion`, `perceived_training_success`, `perceived_recovery`

Questa combinazione di dati oggettivi (GPS/distanza) e soggettivi (autovalutazione dell'atleta) è lo standard nel monitoraggio moderno della scienza dello sport.

---

## 2. Distribuzione del Target: La Sfida dello Sbilanciamento

In [ ]:
target_counts = df[INJURY_COL].value_counts()
target_pct = df[INJURY_COL].value_counts(normalize=True) * 100

print("Target distribution:")
for label in [0, 1]:
    name = "No Injury" if label == 0 else "Injury"
    print(f"  {name} ({label}): {target_counts[label]:,} ({target_pct[label]:.2f}%)")
print(f"\nImbalance ratio: {target_counts[0] / target_counts[1]:.0f}:1")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))

labels = [f"No Injury — 0 ({target_pct[0]:.2f}%)", f"Injury — 1 ({target_pct[1]:.2f}%)"]
colors = [INJURY_PALETTE[0], INJURY_PALETTE[1]]
counts = [target_counts[0], target_counts[1]]

bars = ax.barh(labels, counts, color=colors, edgecolor="white", height=0.5)

for bar, count in zip(bars, counts):
    ax.text(
        bar.get_width() + max(counts) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"n = {count:,}",
        va="center",
        fontweight="bold",
    )

ax.set_xlabel("Number of observations")
ax.set_title("Target Distribution — Day Approach", fontweight="bold")
ax.set_xlim(0, max(counts) * 1.15)
sns.despine(left=True)

save_figure(fig, "01_target_distribution", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretazione

Lo sbilanciamento delle classi è **estremo**: circa un rapporto **72:1** tra campioni negativi e positivi. Questo riflette la realtà — l'infortunio è un evento raro anche tra atleti d'élite che si allenano al limite fisiologico.

**Implicazioni per la modellazione:**
- **L'accuratezza è priva di significato.** Un modello naive che predice "mai infortunato" raggiunge ~98.6% di accuratezza pur essendo completamente inutile. Useremo l'**AUC-ROC** come metrica primaria e l'**AUC-PR** (Precision-Recall) come metrica secondaria, più sensibile alla classe minoritaria.
- Il **class weighting** (`class_weight='balanced'` o `scale_pos_weight`) sarà la nostra strategia primaria per gestire lo sbilanciamento (ADR-003). Questo dice al modello di penalizzare più pesantemente gli infortuni mal classificati, in proporzione alla loro rarità.
- **SMOTE** (Synthetic Minority Oversampling) sarà testato come esperimento secondario, applicato solo all'interno dei fold di training per prevenire il data leakage.

---

## 3. Analisi per Atleta: Chi Si Infortuna?

Poiché i dati provengono da 74 atleti individuali, comprendere la **variabilità inter-atleta** è fondamentale. Gli atleti differiscono per volume di allenamento, suscettibilità agli infortuni e durata del periodo di osservazione.

In [ ]:
athlete_stats = (
    df.groupby(ATHLETE_ID_COL)
    .agg(
        n_observations=(INJURY_COL, "count"),
        n_injuries=(INJURY_COL, "sum"),
        injury_rate=(INJURY_COL, "mean"),
    )
    .sort_values("n_observations", ascending=False)
    .reset_index()
)

athlete_stats["injury_rate_pct"] = athlete_stats["injury_rate"] * 100

print(f"Number of unique athletes: {athlete_stats.shape[0]}")
print("\nObservations per athlete:")
print(f"  Mean:   {athlete_stats['n_observations'].mean():.0f}")
print(f"  Median: {athlete_stats['n_observations'].median():.0f}")
print(f"  Min:    {athlete_stats['n_observations'].min()}")
print(f"  Max:    {athlete_stats['n_observations'].max()}")
print(f"\nAthletes with zero injuries: {(athlete_stats['n_injuries'] == 0).sum()}")
print(f"Athletes with ≥1 injury:     {(athlete_stats['n_injuries'] > 0).sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

colors = [
    INJURY_PALETTE[1] if n > 0 else INJURY_PALETTE[0]
    for n in athlete_stats["n_injuries"]
]

ax.bar(
    range(len(athlete_stats)),
    athlete_stats["n_observations"],
    color=colors,
    edgecolor="white",
    linewidth=0.5,
)

ax.set_xlabel("Athlete (sorted by number of observations)")
ax.set_ylabel("Number of observations")
ax.set_title(
    "Observations per Athlete (red = has injuries, blue = zero injuries)",
    fontweight="bold",
)
ax.set_xticks(range(0, len(athlete_stats), 5))
sns.despine()

save_figure(fig, "01_observations_per_athlete", subdir="eda", close=False)
plt.show()
plt.close(fig)

In [ ]:
athlete_sorted_by_rate = athlete_stats.sort_values("injury_rate_pct", ascending=True)
overall_mean = df[INJURY_COL].mean() * 100

fig, ax = plt.subplots(figsize=(10, 14))

colors = [
    INJURY_PALETTE[1] if r > 0 else PALETTE["neutral"]
    for r in athlete_sorted_by_rate["injury_rate_pct"]
]

ax.barh(
    range(len(athlete_sorted_by_rate)),
    athlete_sorted_by_rate["injury_rate_pct"],
    color=colors,
    edgecolor="white",
    height=0.7,
)

ax.axvline(overall_mean, color="black", linestyle="--", linewidth=1.2, label=f"Overall mean: {overall_mean:.2f}%")
ax.set_ylabel("Athlete")
ax.set_xlabel("Injury rate (%)")
ax.set_title("Injury Rate per Athlete", fontweight="bold")
ax.legend(loc="lower right")
ax.set_yticks(range(0, len(athlete_sorted_by_rate), 5))
sns.despine()

save_figure(fig, "01_injury_rate_per_athlete", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretazione

Gli atleti differiscono **drammaticamente** sia nel volume di dati che nella frequenza di infortuni:

- **Eterogeneità nel volume di dati**: Alcuni atleti contribuiscono migliaia di osservazioni (anni di monitoraggio giornaliero), mentre altri ne hanno solo poche centinaia (una singola stagione). Questo significa che alcuni atleti dominano il training set.

- **Eterogeneità nel tasso di infortuni**: Alcuni atleti hanno zero infortuni nell'intero periodo di osservazione, mentre altri hanno tassi ben superiori alla media dell'1.36%. Questo probabilmente riflette sia la suscettibilità individuale agli infortuni che differenze nell'intensità di allenamento.

**Implicazione per la modellazione — GroupKFold (ADR-006):**
**Dobbiamo** dividere i dati per atleta, non per riga. Se lo stesso atleta appare sia nel training che nel test set, il modello potrebbe apprendere baseline specifiche dell'atleta (es. "l'atleta X ha sempre uno sforzo elevato") anziché predittori di infortunio generalizzabili. `GroupKFold` garantisce che tutte le osservazioni di un atleta appaiano in esattamente un fold.

---

## 4. Copertura Temporale: Quando Sono Stati Osservati gli Atleti?

Comprendere l'arco temporale dei dati di ogni atleta ci aiuta a valutare se il dataset copre un periodo rappresentativo e se gli eventi di infortunio si concentrano in finestre temporali specifiche.

In [ ]:
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

temporal = (
    df.groupby(ATHLETE_ID_COL)
    .agg(
        first_date=(DATE_COL, "min"),
        last_date=(DATE_COL, "max"),
        n_obs=(INJURY_COL, "count"),
        injury_rate=(INJURY_COL, "mean"),
    )
    .sort_values("first_date")
    .reset_index()
)

temporal["span_days"] = (temporal["last_date"] - temporal["first_date"]).dt.days

print(f"Dataset time span: {temporal['first_date'].min()} → {temporal['last_date'].max()}")
print("\nObservation span per athlete:")
print(f"  Mean:   {temporal['span_days'].mean():.0f} days ({temporal['span_days'].mean() / 365:.1f} years)")
print(f"  Median: {temporal['span_days'].median():.0f} days")
print(f"  Min:    {temporal['span_days'].min()} days")
print(f"  Max:    {temporal['span_days'].max()} days")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))

norm = plt.Normalize(
    vmin=temporal["injury_rate"].min(),
    vmax=temporal["injury_rate"].max(),
)
cmap = plt.cm.RdYlBu_r

for i, row in temporal.iterrows():
    color = cmap(norm(row["injury_rate"]))
    ax.barh(
        i,
        (row["last_date"] - row["first_date"]).days,
        left=row["first_date"],
        color=color,
        edgecolor="white",
        height=0.7,
        linewidth=0.3,
    )

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, label="Injury rate", shrink=0.6)

ax.set_ylabel("Athlete (sorted by first observation date)")
ax.set_xlabel("Date")
ax.set_title("Temporal Coverage per Athlete (color = injury rate)", fontweight="bold")
ax.set_yticks(range(0, len(temporal), 5))
sns.despine()

save_figure(fig, "01_temporal_coverage", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretazione

Gli atleti entrano ed escono dal dataset in momenti diversi — questo è tipico degli **studi longitudinali in scienza dello sport** dove gli atleti si uniscono a programmi, cambiano club o si ritirano in momenti diversi.

Osservazioni chiave:
- **Copertura diseguale**: alcuni atleti hanno anni di dati giornalieri, altri solo pochi mesi. Questo influenza la potenza statistica: gli atleti con più dati forniscono stime più stabili.
- **Ingresso scaglionato**: gli atleti non iniziano tutti alla stessa data, il che significa che il dataset cattura contesti di allenamento e competizione diversi.

Questo giustifica ulteriormente lo **split a livello di atleta** (GroupKFold) rispetto allo split temporale (TimeSeriesSplit). Nelle serie temporali classiche, si divide per data. Ma qui, il raggruppamento strutturale più forte è per atleta — il pattern di allenamento di un atleta è più autocorrelato rispetto al trend generale della popolazione (ADR-006).

---

## 5. Distribuzioni delle Feature: Com'è una Giornata di Allenamento?

Ci concentriamo su **day_0** (il giorno corrente) per comprendere le distribuzioni raw delle feature prima di esaminare i pattern temporali nella finestra di 7 giorni. Day 0 è il punto dati più recente — il più vicino al potenziale evento di infortunio.

In [ ]:
day0_cols = [f"day_0_{feat}" for feat in DAY_FEATURES]

fig, axes = plt.subplots(4, 3, figsize=(16, 16))
axes = axes.flatten()

for i, col in enumerate(day0_cols):
    ax = axes[i]
    data = df[col].dropna()
    ax.hist(data, bins=50, color=PALETTE["primary"], alpha=0.7, edgecolor="white")
    ax.set_title(col.replace("day_0_", ""), fontweight="bold", fontsize=11)
    ax.set_ylabel("Count")
    # Add median line
    median_val = data.median()
    ax.axvline(median_val, color=PALETTE["negative"], linestyle="--", linewidth=1,
               label=f"median={median_val:.2f}")
    ax.legend(fontsize=8)

# Hide unused subplots
for j in range(len(day0_cols), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Feature Distributions — Day 0 (Current Day)", fontweight="bold", fontsize=14, y=1.01)
fig.tight_layout()

save_figure(fig, "01_day0_feature_distributions", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretazione

Le distribuzioni delle feature rivelano il **comportamento di allenamento** dei runner d'élite:

- **Feature di volume** (`nr_sessions`, `total_km`): asimmetriche a destra con un grande picco a zero. Questo ha senso — i giorni di riposo (nessun allenamento) sono comuni anche per gli atleti d'élite. Nei giorni attivi, la maggior parte delle sessioni è di distanza moderata.

- **Feature di intensità** (`km_z3_4`, `km_z5_t1_t2`, `km_sprinting`): **fortemente zero-inflate**. Il lavoro ad alta intensità viene svolto con parsimonia — un tratto distintivo dell'allenamento polarizzato, dove ~80% delle sessioni sono facili e solo ~20% sono intense. Questo è il paradigma di allenamento dominante negli sport di endurance.

- **Cross-training** (`strength_training`, `hours_alternative`): simili a binarie o quasi dominate da zero. Il lavoro di forza e l'allenamento alternativo (ciclismo, nuoto, ecc.) sono attività supplementari.

- **Feature soggettive** (`perceived_exertion`, `perceived_training_success`, `perceived_recovery`): distribuzioni più continue, ma si noti la presenza di **valori sentinel (−0.01)** che analizzeremo nella prossima sezione.

**Implicazione per la modellazione**: la forte zero-inflazione significa che i **modelli tree-based** (Random Forest, XGBoost) possono gestire naturalmente la separazione zero/non-zero come decisione binaria ad ogni nodo. I modelli lineari potrebbero avere difficoltà senza feature engineering esplicita.

---

## 6. Analisi del Valore Sentinel: Decodificare i Giorni di Riposo

Il dataset usa **−0.01** come valore sentinel per indicare "nessun dato registrato" — tipicamente un giorno di riposo senza valutazione soggettiva. Comprendere la sua prevalenza tra feature e blocchi temporali è fondamentale per il preprocessing.

**Perché non usare semplicemente 0?** Perché un valore zero nelle feature di allenamento ha un significato reale (es. 0 km corsi), mentre −0.01 segnala esplicitamente "questo dato non è stato raccolto". Per le metriche percettive (sforzo, recupero), se l'atleta non si è allenato, non c'è valutazione soggettiva da riportare.

In [ ]:
sentinel_pct = pd.DataFrame(
    index=range(N_DAY_BLOCKS),
    columns=DAY_FEATURES,
    dtype=float,
)

for block in range(N_DAY_BLOCKS):
    for feat in DAY_FEATURES:
        col = f"day_{block}_{feat}"
        sentinel_pct.loc[block, feat] = (df[col] == SENTINEL_VALUE).mean() * 100

print("Sentinel value (−0.01) prevalence (%) by feature and day block:\n")
print(sentinel_pct.round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

sns.heatmap(
    sentinel_pct.astype(float),
    annot=True,
    fmt=".1f",
    cmap="Blues",
    cbar_kws={"label": "% of rows with sentinel (−0.01)"},
    linewidths=0.5,
    ax=ax,
)

ax.set_xlabel("Feature")
ax.set_ylabel("Day block")
ax.set_title("Sentinel Value (−0.01) Prevalence by Feature × Day Block", fontweight="bold")
ax.set_yticklabels([f"day_{i}" for i in range(N_DAY_BLOCKS)], rotation=0)
ax.set_xticklabels(DAY_FEATURES, rotation=45, ha="right")

save_figure(fig, "01_sentinel_heatmap", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretazione

L'analisi sentinel conferma le nostre aspettative:

- **I valori sentinel appaiono esclusivamente nelle feature percettive/soggettive**: `perceived_exertion`, `perceived_training_success` e `perceived_recovery`. Le feature di volume e intensità usano zeri reali per i giorni di riposo.

- **Coerenza tra i blocchi giornalieri**: la percentuale di sentinel è approssimativamente uniforme tra i giorni 0–6, indicando che i pattern dei giorni di riposo sono stabili all'interno della finestra di 7 giorni.

- **Prevalenza**: una frazione significativa delle osservazioni contiene sentinel, riflettendo che gli atleti fanno regolarmente giorni di riposo (come atteso in qualsiasi programma di allenamento).

**Decisione di preprocessing (ADR-007)**: Sostituiamo −0.01 → 0.0. Il ragionamento è diretto: se un atleta non si è allenato in un dato giorno, il suo sforzo percepito è zero (nessuno sforzo), il suo recupero percepito è zero (nessun allenamento da cui recuperare) e il suo successo percepito dell'allenamento è zero (nessun allenamento da valutare). Questo è il valore semanticamente corretto, e previene che l'artificiale −0.01 introduca rumore nelle analisi basate su distanza e correlazione.

---

## 7. Struttura delle Correlazioni: Ridondanza e Segnale

Analizziamo le correlazioni a due livelli:
1. **Intra-giorno (day 0)** — quali feature co-variano nello stesso giorno?
2. **Tra blocchi giornalieri** — quanto è correlata la stessa feature tra giorni adiacenti?

**Importante**: prima sostituiamo i valori sentinel per evitare correlazioni distorte.

In [ ]:
# Clean sentinel values before correlation analysis
df_clean = handle_sentinel_values(df)

# --- Within-day correlation (day 0 features) ---
day0_data = df_clean[[f"day_0_{feat}" for feat in DAY_FEATURES]]
day0_data.columns = DAY_FEATURES  # short names for readability
corr_day0 = day0_data.corr()

fig, ax = plt.subplots(figsize=(10, 8))

mask = np.triu(np.ones_like(corr_day0, dtype=bool), k=1)
sns.heatmap(
    corr_day0,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax,
)

ax.set_title("Feature Correlation Matrix — Day 0", fontweight="bold")

save_figure(fig, "01_day0_correlation_matrix", subdir="eda", close=False)
plt.show()
plt.close(fig)

In [ ]:
# --- Cross-block correlation for total_km ---
total_km_cols = [f"day_{i}_total_km" for i in range(N_DAY_BLOCKS)]
cross_block = df_clean[total_km_cols].corr()
cross_block.index = [f"day_{i}" for i in range(N_DAY_BLOCKS)]
cross_block.columns = [f"day_{i}" for i in range(N_DAY_BLOCKS)]

fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    cross_block,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax,
)

ax.set_title("Cross-Block Correlation — total_km (Day 0 to Day 6)", fontweight="bold")

save_figure(fig, "01_cross_block_correlation_total_km", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretazione

**Correlazioni intra-giorno (day 0):**
- **Le feature di volume si raggruppano**: `total_km`, `km_z3_4`, `km_z5_t1_t2` sono positivamente correlate — un giorno ad alto volume tende a includere più lavoro di intensità.
- **Le metriche soggettive formano il proprio cluster**: `perceived_exertion`, `perceived_training_success` e `perceived_recovery` sono intercorrelate, riflettendo che le autovalutazioni degli atleti tendono a muoversi insieme.
- **Legami tra cluster**: `perceived_exertion` correla con le feature di volume (le sessioni più dure vengono percepite come più dure), mentre `perceived_recovery` può mostrare correlazioni più deboli o inverse.

**Correlazioni tra blocchi (total_km tra i giorni):**
- **L'autocorrelazione temporale decade con la distanza**: day 0 e day 1 sono più correlati rispetto a day 0 e day 6. Questo riflette la reale **periodizzazione** dell'allenamento — gli atleti strutturano il loro allenamento in pattern (es. duro-facile-duro o periodizzazione a blocchi), creando dipendenze a breve raggio.
- La finestra di 7 giorni cattura approssimativamente un **microciclo** (un blocco standard nella teoria della periodizzazione), motivo per cui è stato scelto questo periodo di lookback.

**Implicazione per la modellazione**: l'alta correlazione tra blocchi significa che alcune feature portano informazione ridondante. I modelli tree-based gestiscono bene questo aspetto (selezionano il miglior split indipendentemente), ma per la Logistic Regression, la multicollinearità potrebbe gonfiare la varianza dei coefficienti.

---

## 8. Relazioni Feature-Target: Cosa Predice l'Infortunio?

Confrontiamo la distribuzione di ogni feature del day 0 tra osservazioni **infortunate** e **non infortunate**. Dato lo sbilanciamento estremo (1.36% positivi), anche piccoli shift nelle distribuzioni sono significativi.

Utilizziamo violin plot con boxplot sovrapposti per mostrare sia la forma della distribuzione che le statistiche di sintesi.

In [ ]:
n_injured = df_clean[df_clean[INJURY_COL] == 1].shape[0]
n_not_injured = df_clean[df_clean[INJURY_COL] == 0].shape[0]

# Build descriptive string labels → seaborn gets clean string categories,
# no np.int64 key issues, no set_xticklabels needed
label_no_injury = f"No Injury\n(n={n_not_injured:,})"
label_injury = f"Injury\n(n={n_injured:,})"
label_order = [label_no_injury, label_injury]
str_palette = {label_no_injury: INJURY_PALETTE[0], label_injury: INJURY_PALETTE[1]}

df_plot = df_clean.assign(
    injury_label=df_clean[INJURY_COL].map({0: label_no_injury, 1: label_injury})
)

fig, axes = plt.subplots(4, 3, figsize=(18, 18))
axes = axes.flatten()

for i, feat in enumerate(DAY_FEATURES):
    ax = axes[i]
    col = f"day_0_{feat}"

    sns.violinplot(
        data=df_plot,
        x="injury_label",
        y=col,
        hue="injury_label",
        order=label_order,
        hue_order=label_order,
        palette=str_palette,
        legend=False,
        ax=ax,
        inner="box",
        cut=0,
    )

    ax.set_title(feat, fontweight="bold", fontsize=11)
    ax.set_xlabel("")

# Hide unused subplots
for j in range(len(DAY_FEATURES), len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    "Feature Distributions by Injury Status — Day 0",
    fontweight="bold",
    fontsize=14,
    y=1.01,
)
fig.tight_layout()
save_figure(fig, "01_day0_features_by_injury", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretazione

I violin plot rivelano le **relazioni feature-target** che i nostri modelli cercheranno di sfruttare:

- **Sovrapposizione sostanziale**: per la maggior parte delle feature, le distribuzioni dei gruppi infortunati e non infortunati si sovrappongono pesantemente. Questo è atteso — l'infortunio è un fenomeno **multicausale** influenzato da genetica, biomeccanica, psicologia e carico di allenamento simultaneamente. Nessuna singola feature separa nettamente le classi.

- **I piccoli shift contano**: cercare le feature dove mediana, dispersione o coda differiscono tra i gruppi. Ad esempio:
  - **`perceived_recovery`**: se più basso per gli atleti infortunati, suggerisce il sotto-recupero come fattore di rischio — un risultato ben consolidato nella scienza dello sport.
  - **`total_km`** o feature di intensità: se leggermente elevate per gli atleti infortunati, supporta l'ipotesi del "troppo e troppo in fretta".

- **Cautela per il campione ridotto**: il gruppo infortunati contiene molti meno campioni, quindi il suo violin plot è più rumoroso. Non dobbiamo sovrainterpretare differenze apparenti senza validazione statistica nella fase di modellazione.

**Perché il task di previsione è difficile**: il miglior AUC-ROC dell'articolo di 0.724 riflette questa sovrapposizione. La previsione degli infortuni dal solo carico di allenamento è intrinsecamente limitata perché il carico di allenamento è solo un pezzo del puzzle.

---

## 9. La Finestra Mobile: La Prospettiva dell'Allenatore

Ogni riga nel dataset rappresenta la visione di un allenatore degli **ultimi 7 giorni**. Day 0 è oggi, day 6 è sei giorni fa. Tracciando come le feature chiave evolvono in questa finestra — suddivise per esito infortunio — possiamo rilevare **pattern di allenamento pre-infortunio**.

Questo è l'equivalente ML di ciò che uno scienziato dello sport chiamerebbe **analisi del rapporto acuto:cronico del carico (ACWR)** — confrontando il carico di allenamento recente con la baseline.

In [ ]:
key_features = ["total_km", "perceived_exertion", "perceived_recovery"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, feat in zip(axes, key_features):
    for label, color, name in [
        (0, INJURY_PALETTE[0], "No Injury"),
        (1, INJURY_PALETTE[1], "Injury"),
    ]:
        subset = df_clean[df_clean[INJURY_COL] == label]
        means = [subset[f"day_{d}_{feat}"].mean() for d in range(N_DAY_BLOCKS)]
        stds = [subset[f"day_{d}_{feat}"].std() for d in range(N_DAY_BLOCKS)]

        ax.plot(range(N_DAY_BLOCKS), means, color=color, marker="o", label=name, linewidth=2)
        ax.fill_between(
            range(N_DAY_BLOCKS),
            [m - s * 0.1 for m, s in zip(means, stds)],
            [m + s * 0.1 for m, s in zip(means, stds)],
            alpha=0.15,
            color=color,
        )

    ax.set_xlabel("Day block (0 = today, 6 = six days ago)")
    ax.set_ylabel(f"Mean {feat}")
    ax.set_title(feat.replace("_", " ").title(), fontweight="bold")
    ax.legend()
    ax.set_xticks(range(N_DAY_BLOCKS))
    ax.set_xticklabels([f"day_{i}" for i in range(N_DAY_BLOCKS)])
    sns.despine(ax=ax)

fig.suptitle(
    "Training Load Across the 7-Day Window by Injury Status",
    fontweight="bold",
    fontsize=14,
    y=1.03,
)
fig.tight_layout()

save_figure(fig, "01_rolling_window_by_injury", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretazione

Questa visualizzazione è il **showcase dell'expertise di dominio** del notebook. Ecco come leggerla attraverso la lente della scienza dello sport:

- **Pattern `total_km`**: se gli atleti infortunati mostrano distanze elevate nella finestra (specialmente nei giorni 3–6), suggerisce **sovraccarico cumulativo** — troppi giorni intensi senza adeguato recupero. Nella teoria della periodizzazione, questo è un microciclo fallito dove l'atleta non ha rispettato il principio duro-facile.

- **Pattern `perceived_exertion`**: se lo sforzo è costantemente più alto per gli atleti infortunati, suggerisce che si allenavano a un'intensità percepita che eccedeva la loro capacità di recupero. Questo è il classico **segnale di overtraining** che allenatori e scienziati dello sport monitorano quotidianamente.

- **Pattern `perceived_recovery`**: se il recupero è più basso per gli atleti infortunati, è il risultato clinicamente più rilevante — **il sotto-recupero è un predittore di infortunio più forte dell'overtraining** nella letteratura moderna di scienza dello sport. Un atleta che riporta costantemente basso recupero è in uno stato di fatica accumulata.

- **Pattern piatti vs. in trend**: se le linee sono relativamente piatte tra i giorni 0–6, significa che la finestra di 7 giorni cattura uno stato stazionario piuttosto che un cambiamento dinamico. Se c'è un trend (es. recupero in calo dal giorno 6 al giorno 0), cattura la *traiettoria* della fatica, non solo il suo livello.

**Connessione al rapporto acuto:cronico del carico (ACWR):** il framework ACWR, reso popolare da Blanch e Gabbett (2016), confronta il carico di allenamento a breve termine (acuto, ~1 settimana) con quello a lungo termine (cronico, ~4 settimane). Un ACWR > 1.5 ("spike") è associato a un rischio elevato di infortunio. Sebbene questa finestra di 7 giorni non calcoli direttamente l'ACWR, la progressione giorno per giorno nella finestra cattura un concetto simile all'interno di un microciclo.

---

## 10. Insight Chiave e Implicazioni per la Modellazione

### Riepilogo dei risultati

1. **Sbilanciamento estremo delle classi (1.36% positivi)**: l'accuratezza è priva di significato come metrica. Dobbiamo usare **AUC-ROC** (primaria) e **AUC-PR** (secondaria). I modelli con class weighting sono essenziali (ADR-003).

2. **Eterogeneità tra atleti**: sia il volume di dati che i tassi di infortunio variano drammaticamente tra gli atleti. **GroupKFold per ID atleta** è obbligatorio per evitare che il modello memorizzi baseline specifiche dell'atleta anziché apprendere segnali di infortunio generalizzabili (ADR-006).

3. **Feature zero-inflate**: molte feature di allenamento (specialmente zone di intensità e cross-training) sono dominate da zeri. I **modelli tree-based** possono sfruttare naturalmente il confine zero/non-zero, mentre i modelli lineari necessiterebbero di feature engineering.

4. **I valori sentinel codificano i giorni di riposo**: il sentinel −0.01 appare esclusivamente nelle feature percettive/soggettive nei giorni di riposo. La sostituzione con 0.0 è semanticamente corretta — "nessun allenamento" significa "zero sforzo percepito" (ADR-007).

5. **L'autocorrelazione temporale riflette la periodizzazione**: le feature di allenamento sono correlate tra giorni adiacenti, riflettendo pattern reali di periodizzazione. La finestra di 7 giorni cattura approssimativamente un microciclo di allenamento.

6. **Debole separazione feature-target**: le distribuzioni dei gruppi infortunati e non infortunati si sovrappongono sostanzialmente per tutte le feature. Questo spiega il moderato AUC-ROC di 0.724 dell'articolo — la previsione degli infortuni dal solo carico di allenamento è un problema genuinamente difficile perché il carico di allenamento è solo uno tra molti fattori di rischio (genetica, biomeccanica, psicologia, nutrizione, sonno).

### Prossimi passi

→ **Notebook 02**: Esplorare l'approccio settimanale, che scambia granularità temporale per aggregazioni settimanali più ricche e introduce le feature di rapporto km relativi.

→ **Notebook 03**: Implementare il preprocessing (sostituzione sentinel, scaling, split train/test per atleta) basandosi sugli insight di questa EDA.